# Módulo 04 · Aula 04 — Tipagem e Estruturas

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Passei um dicionário onde a função esperava um objeto. O erro só apareceu 200 linhas depois, num `AttributeError` que não dizia nada sobre a causa. Levei duas horas."*

Python é de tipagem **dinâmica**: o tipo é do objeto, não do nome. Isso dá flexibilidade — e tira a rede de segurança que outras linguagens oferecem.

Esta aula é sobre recuperar a rede **sem** perder a flexibilidade.

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | Type hints | Documentação que a ferramenta lê |
| 2 | `mypy` | Encontrar erros de tipo **antes** de rodar |
| 3 | **`dataclass`** | Classes de dados em 3 linhas |
| 4 | `frozen`, `slots`, `field` | Imutabilidade e performance |
| 5 | `NamedTuple` e `TypedDict` | Alternativas leves |
| 6 | **Pydantic** | Validação em tempo de execução |
| 7 | Ambientes virtuais | Isolamento de dependências |
| 8 | `pyproject.toml` e `src/` | Organização profissional |

## 1. Type hints

```python
def calcular_frete(valor: float, uf: str) -> float:
    ...
```

**Fato crucial:** Python **ignora** as anotações em tempo de execução. Nada impede você de passar uma string onde a anotação diz `float`.

Então por que usar?

| Ganho | Como |
|-------|------|
| **Autocompletar** | O VS Code sabe o que o objeto tem |
| **Erros antes de rodar** | `mypy` verifica estaticamente |
| **Documentação viva** | Não desatualiza como um comentário |
| **Refatoração segura** | A ferramenta encontra os usos |
| **Validação real** | Pydantic e FastAPI **usam** as anotações |

In [ ]:
def dobrar_estoque(quantidade: int) -> int:
    """A anotação diz int. Python não verifica nada."""
    return quantidade * 2


print("Com int :", dobrar_estoque(5))
print("Com str :", dobrar_estoque("5"), "  ← 😬 não deu erro, fez OUTRA COISA")
print()
print("Anotações guardadas:", dobrar_estoque.__annotations__)
print("Tipo do retorno com str:", type(dobrar_estoque("5")).__name__)

### O vocabulário de tipos

In [ ]:
from typing import Optional, Union, Any, Callable, Literal, TypeVar

# ── Básicos ──
nome: str = "Aurora"
quantidade: int = 42
preco: float = 2599.90
ativo: bool = True

# ── Coleções (Python 3.9+: use os tipos embutidos, não typing.List) ──
cidades: list[str] = ["Campinas", "São Paulo"]
precos: dict[str, float] = {"NB-01": 2599.90}
coordenadas: tuple[float, float] = (-22.9, -47.06)
tags: set[str] = {"eletrônicos", "promoção"}

# ── Tupla de tamanho variável ──
valores: tuple[int, ...] = (1, 2, 3, 4)

# ── Opcional: pode ser None ──
telefone: Optional[str] = None          # forma clássica
telefone2: str | None = None            # ✅ Python 3.10+, prefira esta

# ── União ──
identificador: int | str = "ABC-123"

# ── Aninhados ──
registros: list[dict[str, str | int | float]] = [
    {"sku": "NB-01", "preco": 2599.90, "estoque": 14}
]

# ── Função como parâmetro ──
transformador: Callable[[float], str] = lambda x: f"R$ {x:,.2f}"

# ── Valores literais fechados ──
Status = Literal["pago", "pendente", "cancelado"]
status: Status = "pago"

print("✅ Todas as anotações são válidas")
print(f"registros: {registros}")
print(f"transformador(1234.5): {transformador(1234.5)}")

> ⚠️ **`Any` desliga a verificação.** `def f(x: Any) -> Any` é o mesmo que não anotar. Use apenas quando o tipo for genuinamente qualquer coisa — e prefira um `TypeVar` quando o tipo de saída depender do de entrada.

In [ ]:
# TypeVar: preserva o tipo entre entrada e saída
T = TypeVar("T")


def primeiro(itens: list[T]) -> T | None:
    """A ferramenta sabe: lista de str devolve str; de int devolve int."""
    return itens[0] if itens else None


def com_any(itens: list[Any]) -> Any:
    """A ferramenta não sabe nada."""
    return itens[0] if itens else None


print(primeiro(["a", "b"]), primeiro([1, 2]))

In [ ]:
# Aliases de tipo deixam assinaturas longas legíveis
Registro = dict[str, str | int | float]
Agregacao = dict[str, dict[str, float]]


def agregar(registros: list[Registro], campo: str) -> Agregacao:
    """Compare esta assinatura com a versão sem alias."""
    resultado: Agregacao = {}
    for r in registros:
        chave = str(r[campo])
        resultado.setdefault(chave, {"total": 0.0, "contagem": 0.0})
        resultado[chave]["total"] += float(r.get("preco", 0))
        resultado[chave]["contagem"] += 1
    return resultado


print(agregar([{"cidade": "Campinas", "preco": 100.0},
               {"cidade": "Campinas", "preco": 250.0},
               {"cidade": "Sorocaba", "preco": 80.0}], "cidade"))

## 2. `mypy` — a verificação estática

```bash
pip install mypy
mypy src/
```

O `mypy` lê seu código **sem executá-lo** e aponta incompatibilidades de tipo.

Configure em `pyproject.toml`:

```toml
[tool.mypy]
python_version = "3.11"
warn_return_any = true
warn_unused_configs = true
disallow_untyped_defs = true      # exige anotação em toda função
strict = true                     # o pacote completo
```

> 💡 **Comece frouxo.** Ativar `strict = true` num projeto existente gera centenas de erros e desmotiva. Comece sem flags, corrija o que aparecer, e vá apertando módulo por módulo.

In [ ]:
%%writefile exemplo_tipos.py
"""Arquivo com erros de tipo PROPOSITAIS, para o mypy encontrar."""


def calcular_total(quantidade: int, preco: float) -> float:
    return quantidade * preco


def formatar(valor: float) -> str:
    return f"R$ {valor:,.2f}"


# ❌ ERRO 1: passando str onde espera int
total = calcular_total("3", 249.00)

# ❌ ERRO 2: usando o retorno float como se fosse str
resultado: str = calcular_total(3, 249.00)

# ❌ ERRO 3: chamando método de str num float
texto = formatar(100.0).upper()      # este está certo
numero = calcular_total(2, 50.0).upper()   # ❌ float não tem .upper()

# ❌ ERRO 4: None onde não é permitido
def buscar(id: int) -> str:
    if id > 0:
        return "encontrado"
    # falta o return — devolve None implicitamente

In [ ]:
# Roda o mypy se estiver instalado; se não, explica o que apareceria.
import subprocess
import sys

r = subprocess.run([sys.executable, "-m", "mypy", "exemplo_tipos.py",
                    "--no-color-output", "--no-error-summary"],
                   capture_output=True, text=True)

if r.returncode == 0 and not r.stdout.strip():
    print("ℹ️  mypy não está instalado neste ambiente.")
    print("   Instale com:  pip install mypy")
    print()
    print("   O que ele apontaria no arquivo acima:")
    print('   exemplo_tipos.py:14: error: Argument 1 has incompatible type "str"; expected "int"')
    print('   exemplo_tipos.py:17: error: Incompatible types in assignment (expression has type "float", variable has type "str")')
    print('   exemplo_tipos.py:21: error: "float" has no attribute "upper"')
    print('   exemplo_tipos.py:24: error: Missing return statement')
else:
    print(r.stdout or r.stderr)

> 💭 **Repare no que o mypy faz por você:** cada um desses quatro erros só apareceria em produção, possivelmente semanas depois, num traceback distante da causa. A verificação estática os encontra em segundos, apontando a linha exata.
>
> No **Módulo 09** você vai colocar o `mypy` no CI — e o pipeline vai barrar o merge quando alguém quebrar um tipo.

## 3. `dataclass` — o fim do código repetitivo

Lembra da aula anterior? 25 linhas para uma classe que só carrega dados.

`@dataclass` gera automaticamente `__init__`, `__repr__` e `__eq__` a partir das anotações.

In [ ]:
# ❌ ANTES: 22 linhas
class ProdutoManual:
    def __init__(self, sku, nome, preco, custo, estoque=0, ativo=True):
        self.sku = sku
        self.nome = nome
        self.preco = preco
        self.custo = custo
        self.estoque = estoque
        self.ativo = ativo

    def __repr__(self):
        return (f"ProdutoManual(sku={self.sku!r}, nome={self.nome!r}, "
                f"preco={self.preco!r}, custo={self.custo!r}, "
                f"estoque={self.estoque!r}, ativo={self.ativo!r})")

    def __eq__(self, outro):
        if not isinstance(outro, ProdutoManual):
            return NotImplemented
        return (self.sku, self.nome, self.preco, self.custo,
                self.estoque, self.ativo) == (outro.sku, outro.nome, outro.preco,
                                              outro.custo, outro.estoque, outro.ativo)


print(ProdutoManual("NB-01", "Notebook", 2599.90, 2120.00, 14))

In [ ]:
# ✅ DEPOIS: 8 linhas, mesmo resultado
from dataclasses import dataclass, field, asdict, astuple, replace


@dataclass
class Produto:
    sku: str
    nome: str
    preco: float
    custo: float
    estoque: int = 0
    ativo: bool = True


p = Produto("NB-01", "Notebook Dell", 2599.90, 2120.00, 14)
print(p)
print("Igualdade por valor:", p == Produto("NB-01", "Notebook Dell", 2599.90, 2120.00, 14))

### Os parâmetros do `@dataclass`

| Parâmetro | Efeito |
|-----------|--------|
| `frozen=True` | **Imutável** — atribuir levanta erro. Torna hasheável. |
| `order=True` | Gera `__lt__`, `__le__`, `__gt__`, `__ge__` — permite `sorted()` |
| `slots=True` | Usa `__slots__`: menos memória, atributos fixos (3.10+) |
| `kw_only=True` | Todos os campos viram só-nomeados (3.10+) |
| `eq=False` | Não gera `__eq__` (usa identidade) |

In [ ]:
@dataclass(frozen=True, order=True, slots=True)
class Preco:
    valor: float
    moeda: str = "BRL"


a, b = Preco(100.0), Preco(250.0)

print("ordenação:", sorted([b, a]))
print("hasheável:", {a, b, Preco(100.0)})

try:
    a.valor = 999
except Exception as erro:
    print(f"\n❌ frozen: {type(erro).__name__}: {erro}")

In [ ]:
# Economia de memória com slots
import sys

@dataclass
class SemSlots:
    a: int; b: int; c: int; d: int

@dataclass(slots=True)
class ComSlots:
    a: int; b: int; c: int; d: int


s1, s2 = SemSlots(1, 2, 3, 4), ComSlots(1, 2, 3, 4)
tam1 = sys.getsizeof(s1) + sys.getsizeof(s1.__dict__)
tam2 = sys.getsizeof(s2)

print(f"sem slots: {tam1:>4} bytes  (objeto + __dict__)")
print(f"com slots: {tam2:>4} bytes")
print(f"economia : {(1 - tam2/tam1):.0%}")
print("\n💡 Irrelevante para 10 objetos. Decisivo para 10 milhões.")

### 🔴 `field(default_factory=...)` — a armadilha do mutável

Você já viu isso duas vezes: no argumento padrão (aula 01_04) e no atributo de classe (aula 04_03). Aqui está a terceira encarnação.

**A dataclass te protege:** ela **recusa** um mutável como padrão.

In [ ]:
try:
    @dataclass
    class PedidoErrado:
        id: int
        itens: list = []          # 🔴
except ValueError as erro:
    print(f"❌ {erro}")
    print("\n💡 A dataclass detecta e impede. Use default_factory:")

In [ ]:
@dataclass
class Pedido:
    id: int
    cliente: str
    itens: list[str] = field(default_factory=list)       # ✅ lista NOVA por instância
    metadados: dict = field(default_factory=dict)
    criado_em: str = field(default_factory=lambda: "2026-08-12")

    # Campos que NÃO entram no __init__ nem no repr
    total: float = field(default=0.0, init=False, repr=False)
    _cache: dict = field(default_factory=dict, init=False, repr=False, compare=False)

    def __post_init__(self):
        """Roda logo depois do __init__ gerado. Ideal para derivados e validação."""
        if not self.cliente.strip():
            raise ValueError("cliente não pode ser vazio")
        self.cliente = self.cliente.strip().title()
        self.total = len(self.itens) * 100.0


p1 = Pedido(1, "  maria souza  ", ["Notebook", "Mouse"])
p2 = Pedido(2, "joão lima")

print(p1)
print(p2)
print(f"\nListas independentes? {p1.itens is not p2.itens}")
print(f"total calculado no __post_init__: {p1.total}")

try:
    Pedido(3, "   ")
except ValueError as erro:
    print(f"\n❌ validação: {erro}")

### Utilitários das dataclasses

In [ ]:
p = Pedido(10, "Ana Costa", ["SSD", "Monitor"])

print("asdict :", asdict(p))
print("\nastuple:", astuple(p))

# replace: cria uma CÓPIA com alterações (essencial para frozen)
p_novo = replace(p, cliente="Bruno Rocha")
print(f"\noriginal: {p.cliente}")
print(f"cópia   : {p_novo.cliente}")
print(f"objetos distintos? {p is not p_novo}")

## 4. `NamedTuple` e `TypedDict`

Nem toda estrutura precisa ser uma dataclass.

| | `NamedTuple` | `dataclass` | `TypedDict` |
|---|-------------|-------------|-------------|
| É um | `tuple` | objeto | `dict` |
| Mutável | ❌ | ✅ (salvo `frozen`) | ✅ |
| Desempacota | ✅ `a, b = ponto` | ❌ | ❌ |
| Acesso | `.campo` e `[0]` | `.campo` | `["campo"]` |
| Métodos | ✅ | ✅ | ❌ |
| Uso típico | Retorno múltiplo, chave de dict | Modelo de domínio | JSON tipado |

In [ ]:
from typing import NamedTuple, TypedDict


class Coordenada(NamedTuple):
    latitude: float
    longitude: float

    def distancia_de(self, outra: "Coordenada") -> float:
        return round(((self.latitude - outra.latitude) ** 2 +
                      (self.longitude - outra.longitude) ** 2) ** 0.5, 4)


campinas = Coordenada(-22.9099, -47.0626)
sao_paulo = Coordenada(-23.5505, -46.6333)

print(campinas)
print("por atributo:", campinas.latitude)
print("por índice  :", campinas[0])
lat, lon = campinas                       # ← desempacota, como tupla
print("desempacotado:", lat, lon)
print("distância:", campinas.distancia_de(sao_paulo))
print("é tupla?", isinstance(campinas, tuple))
print("como chave de dict:", {campinas: "Campinas"}[campinas])

In [ ]:
# TypedDict: descreve a FORMA de um dicionário (típico de JSON de API)
class RegistroVenda(TypedDict):
    id: int
    cidade: str
    produto: str
    quantidade: int
    preco: float


class RegistroVendaOpcional(TypedDict, total=False):
    """total=False torna todos os campos opcionais."""
    cupom: str
    observacao: str


def processar(venda: RegistroVenda) -> float:
    """O mypy sabe que venda["quantidade"] é int."""
    return round(venda["quantidade"] * venda["preco"], 2)


registro: RegistroVenda = {
    "id": 1001, "cidade": "Campinas", "produto": "Notebook",
    "quantidade": 2, "preco": 2599.90,
}
print(processar(registro))
print("Em execução, é só um dict:", type(registro).__name__)

> 💡 **Quando usar `TypedDict`?** Quando o dado **já é** um dicionário e você não quer convertê-lo em objeto — típico de resposta de API ou linha de `csv.DictReader`. Você ganha autocompletar e verificação sem custo em execução.
>
> Se o dado é do **seu domínio** e você vai adicionar comportamento, use `dataclass`.

## 5. Pydantic — validação em execução

Type hints não validam nada. **Pydantic valida.**

Ele lê as anotações e, ao criar o objeto, **verifica e converte** os dados. É a biblioteca que sustenta o FastAPI (Módulo 06).

| | `dataclass` | Pydantic |
|---|-------------|----------|
| Valida em execução | ❌ | ✅ |
| Converte tipos | ❌ | ✅ (`"42"` → `42`) |
| Regras customizadas | Manual | `@field_validator` |
| Serialização JSON | Manual | `.model_dump_json()` |
| Custo | Zero (biblioteca padrão) | Dependência externa |

In [ ]:
# Instala o Pydantic se ainda não estiver disponível
try:
    import pydantic
    print(f"✅ Pydantic {pydantic.VERSION} já instalado")
except ImportError:
    print("Instalando pydantic...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "pydantic", "--quiet"],
                   check=False)
    try:
        import pydantic
        print(f"✅ Pydantic {pydantic.VERSION} instalado")
    except ImportError:
        print("⚠️  Não foi possível instalar. As células seguintes serão puladas.")
        pydantic = None

In [ ]:
# A diferença fundamental, lado a lado
from dataclasses import dataclass


@dataclass
class ProdutoDataclass:
    sku: str
    preco: float
    estoque: int


# A dataclass ACEITA tudo — a anotação é decorativa
ruim = ProdutoDataclass(sku=123, preco="muito caro", estoque=[1, 2, 3])
print("dataclass aceitou:", ruim)
print("tipo de preco:", type(ruim.preco).__name__, "  ← 😬")

In [ ]:
if pydantic:
    from pydantic import BaseModel, Field, field_validator, ValidationError

    class ProdutoPydantic(BaseModel):
        sku: str = Field(min_length=5, max_length=20)
        nome: str = Field(min_length=3)
        preco: float = Field(gt=0, description="Preço de venda em reais")
        custo: float = Field(ge=0)
        estoque: int = Field(default=0, ge=0)
        ativo: bool = True

        @field_validator("sku")
        @classmethod
        def sku_maiusculo(cls, v: str) -> str:
            """Validadores também TRANSFORMAM."""
            if not v.replace("-", "").isalnum():
                raise ValueError("SKU só aceita letras, números e hífen")
            return v.upper()

        @field_validator("custo")
        @classmethod
        def custo_menor_que_preco(cls, v, info):
            if "preco" in info.data and v > info.data["preco"]:
                raise ValueError(f"custo {v} maior que o preço {info.data['preco']}")
            return v

    # ✅ Dado válido: converte tipos automaticamente
    p = ProdutoPydantic(
        sku="nb-dell-15",          # vira maiúsculo
        nome="Notebook Dell",
        preco="2599.90",           # str → float
        custo=2120,                # int → float
        estoque="14",              # str → int
    )
    print(p)
    print(f"\ntipo de preco : {type(p.preco).__name__}")
    print(f"tipo de estoque: {type(p.estoque).__name__}")
    print(f"sku normalizado: {p.sku}")

In [ ]:
if pydantic:
    # ❌ Dados inválidos: erros ESTRUTURADOS, todos de uma vez
    try:
        ProdutoPydantic(sku="ab", nome="X", preco=-10, custo=99999, estoque=-5)
    except ValidationError as erro:
        print(f"{erro.error_count()} erro(s) encontrado(s):\n")
        for e in erro.errors():
            campo = ".".join(str(x) for x in e["loc"])
            print(f"  {campo:<10} {e['msg']}")

> 💡 **Repare: o Pydantic reporta todos os erros de uma vez.** Uma validação manual com `if/raise` para no primeiro. Para um formulário ou uma API, mostrar os quatro problemas juntos é muito melhor que fazer o usuário descobrir um por vez.
>
> 💭 **Por que o `custo=99999` não apareceu na lista?** Porque o validador `custo_menor_que_preco` compara com `info.data["preco"]` — e `preco` já tinha falhado, então não está em `info.data`. O Pydantic não valida contra um campo que ele sabe estar inválido.
>
> Esse comportamento é deliberado e correto: reportar "custo maior que o preço" quando o preço nem é válido só confundiria. Mas é bom saber que **validadores dependentes de outros campos só rodam se aqueles passarem**.

In [ ]:
if pydantic:
    # Modelos aninhados e serialização
    from pydantic import BaseModel

    class Endereco(BaseModel):
        cidade: str
        uf: str = Field(min_length=2, max_length=2)
        cep: str = Field(pattern=r"^\d{5}-?\d{3}$")

        @field_validator("uf")
        @classmethod
        def uf_maiuscula(cls, v: str) -> str:
            return v.upper()

    class Cliente(BaseModel):
        nome: str
        email: str = Field(pattern=r"^[^@]+@[^@]+\.[^@]+$")
        endereco: Endereco                    # ← modelo aninhado
        tags: list[str] = Field(default_factory=list)

    c = Cliente(
        nome="Maria Souza",
        email="maria@aurora.com.br",
        endereco={"cidade": "Campinas", "uf": "sp", "cep": "13000-000"},  # dict vira Endereco!
        tags=["vip", "recorrente"],
    )
    print(c)
    print(f"\nTipo do endereço: {type(c.endereco).__name__}")
    print(f"Acesso aninhado : {c.endereco.uf}")
    print(f"\nJSON:\n{c.model_dump_json(indent=2)}")

In [ ]:
if pydantic:
    # Validando dados sujos de CSV — o caso do Atlas
    from pydantic import ValidationError

    class LinhaVenda(BaseModel):
        id: int
        cidade: str
        quantidade: int = Field(gt=0)
        preco: float = Field(ge=0)
        status: str

        @field_validator("cidade")
        @classmethod
        def normalizar_cidade(cls, v: str) -> str:
            return " ".join(v.split()).title()

        @field_validator("status")
        @classmethod
        def validar_status(cls, v: str) -> str:
            v = v.strip().lower()
            if v not in {"pago", "pendente", "cancelado"}:
                raise ValueError(f"status inválido: {v}")
            return v

    linhas_sujas = [
        {"id": "1001", "cidade": "  campinas  ", "quantidade": "2", "preco": "2599.90", "status": "PAGO"},
        {"id": "1002", "cidade": "são   paulo", "quantidade": "dez", "preco": "89.90", "status": "pago"},
        {"id": "1003", "cidade": "Sorocaba", "quantidade": "-2", "preco": "249.00", "status": "pago"},
        {"id": "1004", "cidade": "Santos", "quantidade": "1", "preco": "1199.00", "status": "entregue"},
        {"id": "1005", "cidade": "Curitiba", "quantidade": "3", "preco": "489.00", "status": "pago"},
    ]

    validas, rejeitadas = [], []
    for numero, linha in enumerate(linhas_sujas, start=2):
        try:
            validas.append(LinhaVenda(**linha))
        except ValidationError as erro:
            motivos = "; ".join(f"{'.'.join(map(str, e['loc']))}: {e['msg']}"
                                for e in erro.errors())
            rejeitadas.append({"linha": numero, "motivo": motivos})

    print(f"✅ {len(validas)} válidas:")
    for v in validas:
        print(f"   {v.id} | {v.cidade:<12} | {v.quantidade}x R$ {v.preco:,.2f} | {v.status}")

    print(f"\n❌ {len(rejeitadas)} rejeitadas:")
    for r in rejeitadas:
        print(f"   linha {r['linha']}: {r['motivo']}")

> 💭 **Compare com o `validacao.py` que você escreveu no Módulo 01.** Aquele arquivo tinha ~150 linhas de conversores, checagens e mensagens de erro. O Pydantic faz o mesmo com um modelo declarativo — e ainda normaliza, converte e serializa.
>
> ⚠️ **Isso não significa que o M01 foi perdido.** Você precisava entender o que a validação faz antes de delegá-la. Quem só sabe usar Pydantic não sabe depurar quando ele se comporta de forma inesperada.

## 6. Ambientes virtuais

Um **venv** é uma pasta com sua própria cópia do Python e suas próprias dependências.

**Por que é obrigatório:** o projeto A precisa do `pandas 1.5`, o B do `pandas 2.1`. Sem isolamento, instalar um quebra o outro.

```bash
python -m venv .venv                    # cria

source .venv/bin/activate               # ativa (macOS/Linux)
.venv\Scripts\Activate.ps1              # ativa (Windows PowerShell)

deactivate                              # desativa
```

In [ ]:
import sys
from pathlib import Path

print("Executável:", sys.executable)
print("Prefixo   :", sys.prefix)
print("Base      :", sys.base_prefix)
print()
if sys.prefix != sys.base_prefix:
    print("✅ Você está DENTRO de um ambiente virtual")
else:
    print("⚠️  Você está usando o Python GLOBAL")
    print("   Para o projeto Atlas, crie e ative um venv.")

### `requirements.txt` vs `pyproject.toml`

| | `requirements.txt` | `pyproject.toml` |
|---|--------------------|------------------|
| O que é | Lista de pacotes | Metadados completos do projeto |
| Padrão | Convenção antiga | **PEP 621 — o padrão moderno** |
| Contém | Só dependências | Deps, build, ferramentas, scripts |
| Configura ferramentas | ❌ | ✅ ruff, mypy, pytest, black |

> 🧭 **Recomendação:** use `pyproject.toml`. O `requirements.txt` ainda aparece em muito projeto legado e em Dockerfiles, mas o padrão moderno é o TOML.

In [ ]:
%%writefile pyproject_exemplo.toml
# ═══════════════════════════════════════════════════════════════
#  pyproject.toml — o arquivo único de configuração do projeto
# ═══════════════════════════════════════════════════════════════

[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "atlas"
version = "0.4.0"
description = "Sistema central da Aurora Comércio"
readme = "README.md"
requires-python = ">=3.10"
authors = [{name = "Aurora Comércio", email = "engenharia@aurora.com.br"}]

dependencies = [
    "pydantic>=2.0",
]

[project.optional-dependencies]
# Instale com: pip install -e ".[dev]"
dev = [
    "pytest>=8.0",
    "pytest-cov",
    "mypy>=1.8",
    "ruff>=0.6",
]

[project.scripts]
# Cria o comando `atlas` no terminal após `pip install -e .`
atlas = "atlas.cli:main"

# ── Layout src/ ──
[tool.setuptools.packages.find]
where = ["src"]

# ═══════════════════════════════════════════════════════════════
#  Configuração das ferramentas — tudo em um arquivo só
# ═══════════════════════════════════════════════════════════════

[tool.ruff]
line-length = 100
target-version = "py310"

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]
# E=pycodestyle, F=pyflakes, I=isort, UP=pyupgrade,
# B=bugbear (bugs prováveis), SIM=simplificações

[tool.mypy]
python_version = "3.10"
warn_return_any = true
warn_unused_configs = true
disallow_untyped_defs = true
# Comece frouxo e vá apertando módulo a módulo:
# [[tool.mypy.overrides]]
# module = "atlas.legado.*"
# disallow_untyped_defs = false

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --cov=atlas --cov-report=term-missing"

### O layout `src/`

```
projeto_Atlas/
├── pyproject.toml
├── src/
│   └── atlas/
│       ├── __init__.py
│       └── ...
└── tests/
```

**Por que a pasta `src/`?** Porque sem ela, `import atlas` funciona por acidente — o Python encontra a pasta local antes do pacote instalado.

Com `src/`, o import **só** funciona se o pacote estiver instalado. Isso garante que você está testando o que o usuário vai receber, não uma cópia local que talvez esteja desatualizada.

```bash
pip install -e .          # instalação editável
pip install -e ".[dev]"   # com as dependências de desenvolvimento
```

> 💡 **Instalação editável (`-e`)** cria um link para o seu código-fonte em vez de copiá-lo. Você edita o arquivo e a mudança vale imediatamente — sem reinstalar. É como se desenvolve um pacote Python.
>
> Depois disso, você pode apagar aquele `sys.path.insert(...)` do `main.py`.

## 🔧 Prática guiada — Os modelos tipados do Atlas

In [ ]:
%%writefile atlas_modelos.py
"""Modelos de domínio do Atlas — dataclasses tipadas.

Estratégia adotada no projeto:

  • dataclass  para modelos INTERNOS (rápidas, sem dependência)
  • Pydantic   para dados de FRONTEIRA (CSV, API, entrada do usuário)

Por quê? Porque validar no meio do sistema é desperdício: se o dado
já entrou validado, revalidar a cada camada só custa tempo.
Valide UMA VEZ, na fronteira, e confie no tipo daí em diante.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Literal


# ═══════════════════════════════════════════════════════════════
#  Enums — domínios fechados
# ═══════════════════════════════════════════════════════════════

class Status(str, Enum):
    """Herdar de str permite comparar com string diretamente."""
    PAGO = "pago"
    PENDENTE = "pendente"
    CANCELADO = "cancelado"


class Canal(str, Enum):
    SITE = "site"
    APP = "app"
    MARKETPLACE = "marketplace"


UF = Literal["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "PE", "CE", "DF", "GO", "ES"]


# ═══════════════════════════════════════════════════════════════
#  Modelos
# ═══════════════════════════════════════════════════════════════

@dataclass(frozen=True, slots=True)
class Produto:
    """Imutável: um produto não muda de identidade.

    frozen=True dá hashabilidade de graça, permitindo usar Produto
    como chave de dicionário nas agregações.
    """
    sku: str
    nome: str
    categoria: str
    preco: float
    custo: float

    def __post_init__(self) -> None:
        if self.preco < 0:
            raise ValueError(f"preço negativo: {self.preco}")
        if self.custo < 0:
            raise ValueError(f"custo negativo: {self.custo}")

    @property
    def margem_unitaria(self) -> float:
        return round(self.preco - self.custo, 2)

    @property
    def margem_percentual(self) -> float:
        return 0.0 if self.preco == 0 else round(self.margem_unitaria / self.preco, 4)


@dataclass(slots=True)
class ItemVenda:
    produto: Produto
    quantidade: int
    preco_unitario: float

    def __post_init__(self) -> None:
        if self.quantidade <= 0:
            raise ValueError(f"quantidade deve ser positiva: {self.quantidade}")

    @property
    def total(self) -> float:
        return round(self.quantidade * self.preco_unitario, 2)

    @property
    def margem(self) -> float:
        return round(self.quantidade * (self.preco_unitario - self.produto.custo), 2)

    @property
    def desconto_praticado(self) -> float:
        """Quanto o preço praticado ficou abaixo do de catálogo."""
        if self.produto.preco == 0:
            return 0.0
        return round(1 - self.preco_unitario / self.produto.preco, 4)


@dataclass(slots=True)
class Pedido:
    id: int
    cliente: str
    cidade: str
    uf: str
    canal: Canal
    status: Status
    data: str
    frete: float = 0.0
    itens: list[ItemVenda] = field(default_factory=list)

    @property
    def subtotal(self) -> float:
        return round(sum(i.total for i in self.itens), 2)

    @property
    def total(self) -> float:
        return round(self.subtotal + self.frete, 2)

    @property
    def margem(self) -> float:
        return round(sum(i.margem for i in self.itens), 2)

    @property
    def quantidade_itens(self) -> int:
        return sum(i.quantidade for i in self.itens)

    @property
    def faturado(self) -> bool:
        return self.status is Status.PAGO

    @property
    def mes(self) -> str:
        return self.data[:7]

    def adicionar(self, item: ItemVenda) -> Pedido:
        self.itens.append(item)
        return self


@dataclass(frozen=True, slots=True)
class Metricas:
    """Resultado de agregação. Imutável: é um retrato de um instante."""
    pedidos: int = 0
    itens: int = 0
    receita: float = 0.0
    margem: float = 0.0

    @property
    def ticket_medio(self) -> float:
        return round(self.receita / self.pedidos, 2) if self.pedidos else 0.0

    @property
    def margem_percentual(self) -> float:
        return round(self.margem / self.receita, 4) if self.receita else 0.0

    def somar(self, pedido: Pedido) -> Metricas:
        """Devolve uma NOVA instância — imutabilidade preservada."""
        return Metricas(
            pedidos=self.pedidos + 1,
            itens=self.itens + pedido.quantidade_itens,
            receita=round(self.receita + pedido.subtotal, 2),
            margem=round(self.margem + pedido.margem, 2),
        )

In [ ]:
import atlas_modelos as m
import random

rnd = random.Random(42)

catalogo = [
    m.Produto("NB-01", "Notebook Dell",   "Notebooks",    2599.90, 2120.00),
    m.Produto("NB-02", "Notebook Acer",   "Notebooks",    3299.00, 2780.00),
    m.Produto("MO-01", "Monitor LG",      "Monitores",    1199.00,  920.00),
    m.Produto("PE-01", "Mouse Logitech",  "Periféricos",    89.90,   52.00),
    m.Produto("PE-02", "Teclado Redragon","Periféricos",   249.00,  150.00),
    m.Produto("AR-01", "SSD 1TB",         "Armazenamento", 489.00,  360.00),
]

print("CATÁLOGO")
print(f"{'SKU':<8}{'Produto':<20}{'Preço':>10}{'Margem':>10}{'%':>8}")
print("─" * 56)
for p in catalogo:
    print(f"{p.sku:<8}{p.nome:<20}{p.preco:>10,.2f}"
          f"{p.margem_unitaria:>10,.2f}{p.margem_percentual:>8.1%}")

In [ ]:
# frozen + slots: imutável e hasheável
try:
    catalogo[0].preco = 9999
except Exception as erro:
    print(f"❌ frozen: {type(erro).__name__}")

print("\nHasheável (serve como chave de dict):")
vendas_por_produto = {p: 0 for p in catalogo}
print(f"   {len(vendas_por_produto)} chaves criadas")

# __post_init__ validando
try:
    m.Produto("XX-01", "Inválido", "Teste", -100, 50)
except ValueError as erro:
    print(f"\n❌ validação: {erro}")

In [ ]:
# Gerando pedidos
cidades = [("Campinas", "SP"), ("São Paulo", "SP"), ("Curitiba", "PR"),
           ("Salvador", "BA"), ("Recife", "PE")]

pedidos: list[m.Pedido] = []
for pid in range(1, 151):
    cidade, uf = rnd.choice(cidades)
    pedido = m.Pedido(
        id=pid,
        cliente=f"Cliente {rnd.randint(1, 30):02d}",
        cidade=cidade, uf=uf,
        canal=rnd.choices(list(m.Canal), weights=[50, 30, 20])[0],
        status=rnd.choices(list(m.Status), weights=[80, 12, 8])[0],
        data=f"2026-{rnd.choice(['05','06','07'])}-{rnd.randint(1,28):02d}",
        frete=rnd.choice([0.0, 9.90, 19.90]),
    )
    for produto in rnd.sample(catalogo, rnd.choices([1, 2, 3], weights=[55, 30, 15])[0]):
        pedido.adicionar(m.ItemVenda(
            produto=produto,
            quantidade=rnd.choices([1, 2, 3, 5], weights=[55, 25, 13, 7])[0],
            preco_unitario=round(produto.preco * rnd.choice([1.0, 1.0, 0.95, 0.90]), 2),
        ))
    pedidos.append(pedido)

print(f"✅ {len(pedidos)} pedidos gerados\n")
exemplo = pedidos[0]
print(f"Pedido {exemplo.id} — {exemplo.cliente} ({exemplo.cidade}/{exemplo.uf})")
print(f"  canal    : {exemplo.canal.value}")
print(f"  status   : {exemplo.status.value}  (faturado? {exemplo.faturado})")
print(f"  mês      : {exemplo.mes}")
print(f"  itens    : {exemplo.quantidade_itens}")
print(f"  subtotal : R$ {exemplo.subtotal:,.2f}")
print(f"  total    : R$ {exemplo.total:,.2f}")
print(f"  margem   : R$ {exemplo.margem:,.2f}")
for item in exemplo.itens:
    print(f"    • {item.produto.nome:<20} {item.quantidade}x "
          f"R$ {item.preco_unitario:>9,.2f}  (desconto {item.desconto_praticado:.0%})")

In [ ]:
# Agregação com o modelo imutável Metricas
from collections import defaultdict

por_uf: dict[str, m.Metricas] = defaultdict(m.Metricas)
for pedido in pedidos:
    if pedido.faturado:
        por_uf[pedido.uf] = por_uf[pedido.uf].somar(pedido)

print(f"{'UF':<5}{'Pedidos':>9}{'Itens':>8}{'Receita':>14}{'Ticket':>12}{'Margem %':>10}")
print("─" * 58)
for uf, met in sorted(por_uf.items(), key=lambda kv: -kv[1].receita):
    print(f"{uf:<5}{met.pedidos:>9}{met.itens:>8}{met.receita:>14,.2f}"
          f"{met.ticket_medio:>12,.2f}{met.margem_percentual:>10.1%}")

> 💭 **Repare no desenho.** `Metricas` é `frozen`, e `somar()` devolve uma **nova** instância em vez de mutar. Isso se chama estrutura de dados **persistente**, e traz duas vantagens: nenhuma parte do código pode alterar suas métricas por acidente, e o objeto é seguro para usar em código concorrente (aula 04_06).
>
> O custo é criar um objeto novo a cada soma. Para 150 pedidos, irrelevante. Para 150 milhões, você usaria um acumulador mutável.

## 📝 Exercícios

**E1.** Anote completamente esta função, incluindo o retorno:
```python
def agrupar(registros, campo, agregador=sum):
    ...
```

**E2.** Crie os aliases `SKU = str`, `Reais = float` e `Registro = dict[str, str | float]`, e use-os numa função de processamento.

**E3.** Escreva `primeiro_ou_padrao(itens, padrao)` usando `TypeVar`, de forma que o mypy saiba o tipo devolvido.

**E4.** Converta esta classe em `dataclass` e diga quantas linhas economizou:
```python
class Cliente:
    def __init__(self, nome, email, cidade, uf, ativo=True):
        ...
    def __repr__(self): ...
    def __eq__(self, o): ...
```

**E5.** Crie `@dataclass(frozen=True, order=True)` chamada `Versao(maior, menor, correcao)` e ordene uma lista de versões.

**E6.** Crie uma dataclass `Carrinho` com `itens: list` e `cupons: dict`, usando `field(default_factory=...)`. Prove que duas instâncias não compartilham as coleções.

**E7.** Use `__post_init__` para validar e normalizar uma dataclass `Endereco` (CEP com 8 dígitos, UF maiúscula, cidade em Title Case).

**E8.** Compare memória e tempo de criação entre `@dataclass` e `@dataclass(slots=True)` com 100.000 instâncias.

**E9.** Crie um `NamedTuple` `Resultado(sucesso, valor, erro)` e use-o como retorno de uma função que pode falhar. Compare com levantar exceção — quando cada abordagem é melhor?

**E10.** Defina um `TypedDict` para a resposta de uma API de CEP (`{"cep", "logradouro", "bairro", "localidade", "uf"}`) e escreva uma função que a processe.

**E11.** Crie um modelo Pydantic `PedidoAPI` com validação de: id positivo, e-mail válido, quantidade entre 1 e 100, status em lista fechada, e data no formato ISO. Teste com 5 entradas inválidas diferentes.

**E12.** Use `@field_validator` para normalizar o nome do cliente (trim + Title Case) e o e-mail (lowercase). Prove que a transformação acontece.

**E13.** Crie modelos Pydantic aninhados: `Pedido` contendo `Cliente` e `list[Item]`. Serialize para JSON e desserialize de volta.

**E14.** Escreva um `pyproject.toml` completo para o Atlas, com dependências, extras de dev, configuração de ruff, mypy e pytest, e um script de console.

**E15.** Converta `atlas_modelos.py` para usar Pydantic no lugar de dataclasses. Meça o custo de criação de 100.000 objetos nos dois casos e discuta o trade-off.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```python
# ── Type hints ──
def f(a: int, b: str = "x", *args: int, **kw: float) -> bool: ...

lista: list[str]                dicionario: dict[str, float]
tupla: tuple[int, str]          variavel: tuple[int, ...]
conjunto: set[str]
opcional: str | None            # 3.10+, prefira a Optional[str]
uniao: int | str
funcao: Callable[[int, str], bool]
literal: Literal["pago", "pendente"]
qualquer: Any                   # ⚠️ desliga a verificação

T = TypeVar("T")
def primeiro(xs: list[T]) -> T | None: ...

Registro = dict[str, str | float]        # alias

# ── mypy ──
# pip install mypy && mypy src/
# [tool.mypy] em pyproject.toml
# comece frouxo, aperte módulo a módulo

# ── dataclass ──
from dataclasses import dataclass, field, asdict, astuple, replace

@dataclass(frozen=True, order=True, slots=True, kw_only=True)
class Modelo:
    obrigatorio: str
    com_padrao: int = 0
    mutavel: list = field(default_factory=list)     # ⚠️ NUNCA = []
    derivado: float = field(default=0.0, init=False, repr=False)

    def __post_init__(self):
        # validação e campos derivados
        ...

asdict(obj)   astuple(obj)   replace(obj, campo=novo)

# ── NamedTuple / TypedDict ──
class Ponto(NamedTuple):
    x: float
    y: float                    # imutável, desempacota, é tupla

class Resposta(TypedDict):
    campo: str                  # é um dict comum em execução

class Opcional(TypedDict, total=False):
    talvez: str

# ── Pydantic ──
from pydantic import BaseModel, Field, field_validator, ValidationError

class Modelo(BaseModel):
    nome: str = Field(min_length=3, max_length=50)
    idade: int = Field(ge=0, le=120)
    preco: float = Field(gt=0)
    email: str = Field(pattern=r"^[^@]+@[^@]+\.[^@]+$")
    tags: list[str] = Field(default_factory=list)

    @field_validator("nome")
    @classmethod
    def normalizar(cls, v: str) -> str:
        return v.strip().title()

obj = Modelo(**dados)              # valida E converte
obj.model_dump()                   # → dict
obj.model_dump_json(indent=2)      # → JSON
Modelo.model_validate_json(texto)  # ← JSON
erro.errors()                      # lista estruturada de problemas

# ── Ambiente ──
python -m venv .venv
source .venv/bin/activate          # .venv\Scripts\Activate.ps1 no Windows
pip install -e ".[dev]"            # editável, com extras
sys.prefix != sys.base_prefix      # está num venv?
```

## ✅ Checklist de saída

- [ ] Sei que type hints não validam em execução — e por que ainda valem
- [ ] Uso `list[str]` em vez de `List[str]`, e `str | None` em vez de `Optional[str]`
- [ ] Crio aliases para assinaturas complexas
- [ ] Evito `Any` e sei quando usar `TypeVar`
- [ ] Sei rodar `mypy` e configurá-lo no `pyproject.toml`
- [ ] **Uso `@dataclass` em vez de escrever `__init__`/`__repr__`/`__eq__`**
- [ ] Sei o efeito de `frozen`, `order`, `slots` e `kw_only`
- [ ] **Uso `field(default_factory=...)` para mutáveis**
- [ ] Uso `__post_init__` para validar e derivar campos
- [ ] Conheço `asdict`, `astuple` e `replace`
- [ ] Escolho entre `dataclass`, `NamedTuple` e `TypedDict` com critério
- [ ] Sei o que o Pydantic faz além dos type hints
- [ ] Escrevo `@field_validator` para normalizar e validar
- [ ] Entendo "validar na fronteira, confiar no miolo"
- [ ] Sempre trabalho dentro de um ambiente virtual
- [ ] Sei o que é `pyproject.toml` e por que o layout `src/` existe
- [ ] Sei o que `pip install -e .` faz

---

### ➡️ Próxima aula

**`04_05_Ecosistema_Avancado.ipynb`** — Datas, logging estruturado e context managers. Onde o Atlas para de usar `print` e passa a ter observabilidade.